In [ ]:
deps_path = '/kaggle/input/datasets/nhhsag12/colpali-dependency'
!pip install --no-index --find-links {deps_path} --requirement {deps_path}/requirements.txt


In [ ]:
import os
import gc
import glob
import json
import pickle
import time
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from tqdm.notebook import tqdm

# ==============================================================================
# CONFIG — follow MMDocIR template, only data flow switched to ViDoRe
# ==============================================================================

INDEX_PKL_PATH = "/kaggle/input/datasets/namthi/vidore-encoded/colpali13_page_index_vidore_v3_pagelevel.pkl"
VIDORE_DATASET_ROOT = "/kaggle/input/datasets/namthi/vidore-v3"

# ColPali model path (same template settings)
COLPALI_BASE   = "/kaggle/input/models/nhhsag12/model-colpali-base/pytorch/default/3"
COLPALI_LORA   = "/kaggle/input/models/nhhsag12/model-colpali/pytorch/default/2"

WORKING_DIR = "/kaggle/working"
os.makedirs(WORKING_DIR, exist_ok=True)

# Optional domain filter for ViDoRe query files; [] means all
DOMAIN_FILTER = []

# Method config (kept from template)
TOPK_RATIOS        = [round(i * 0.1, 1) for i in range(1, 10)]   # 0.1 … 0.9
N_LAST_LAYERS_LIST = [4, 8, 16, 32]   # for Ours/Ablation method
NORMALIZE_MODES    = ['pre', 'post']  # for Ours/Ablation method
ATTN_N_LAYERS_LIST = [1, 2, 4, 8]     # for Attention Score method
KMEANS_ITERS       = 10               # for Spherical KMeans method
N_RANDOM_SEEDS     = 1                # for Random Pruning method (seeds to average)

print("Config loaded.")
print(f"INDEX_PKL_PATH      : {INDEX_PKL_PATH}")
print(f"VIDORE_DATASET_ROOT : {VIDORE_DATASET_ROOT}")
print(f"COLPALI_BASE        : {COLPALI_BASE}")
print(f"COLPALI_LORA        : {COLPALI_LORA}")
print(f"TOPK_RATIOS         : {TOPK_RATIOS}")
print(f"N_LAST_LAYERS_LIST  : {N_LAST_LAYERS_LIST}")
print(f"NORMALIZE_MODES     : {NORMALIZE_MODES}")
print(f"ATTN_N_LAYERS_LIST  : {ATTN_N_LAYERS_LIST}")
print(f"KMEANS_ITERS        : {KMEANS_ITERS}")
print(f"N_RANDOM_SEEDS      : {N_RANDOM_SEEDS}")

In [ ]:
# ==============================================================================
# Load encoded ViDoRe page embeddings (single-file or sharded manifest)
# ==============================================================================

if not os.path.isfile(INDEX_PKL_PATH):
    raise FileNotFoundError(f"Index PKL not found: {INDEX_PKL_PATH}")

with open(INDEX_PKL_PATH, "rb") as f:
    payload = pickle.load(f)

if not isinstance(payload, dict):
    raise ValueError("Expected dict payload in encoded ViDoRe PKL.")

raw_embeddings = []
page_keys = []
meta_records = []

# Support sharded manifest payload to avoid OOM during encoding.
if payload.get("format") == "vidore_sharded_v1":
    shard_files = payload.get("shard_files", [])
    if not shard_files:
        raise ValueError("Sharded payload has no shard_files.")

    print(f"Loading sharded index from {len(shard_files)} shards...")
    for sp in shard_files:
        if not os.path.isfile(sp):
            raise FileNotFoundError(f"Shard not found: {sp}")
        with open(sp, "rb") as sf:
            shard = pickle.load(sf)
        raw_embeddings.extend(shard.get("fused_index", []))
        page_keys.extend(shard.get("page_keys", []))
        meta_records.extend(shard.get("metadata", []))
else:
    raw_embeddings = payload.get("fused_index", payload.get("embeddings", []))
    page_keys = payload.get("page_keys", [])
    meta_records = payload.get("metadata", [])

if not raw_embeddings:
    raise ValueError("No embeddings found in PKL payload.")

meta_df = pd.DataFrame(meta_records) if meta_records else pd.DataFrame()

all_page_embeddings = []
for emb in raw_embeddings:
    if isinstance(emb, torch.Tensor):
        arr = emb.detach().cpu().numpy()
    else:
        arr = np.asarray(emb)
    if arr.ndim != 2:
        raise ValueError(f"Embedding must be 2D (tokens x dim), got {arr.shape}")
    # Keep source dtype if float16/float32 to reduce memory pressure.
    if arr.dtype not in (np.float16, np.float32):
        arr = arr.astype(np.float32, copy=False)
    all_page_embeddings.append(arr)

n_pages = len(all_page_embeddings)
all_page_indices = list(range(n_pages))

if len(meta_df) < n_pages:
    pad_rows = n_pages - len(meta_df)
    meta_df = pd.concat([meta_df, pd.DataFrame(index=range(pad_rows))], ignore_index=True)
meta_df = meta_df.iloc[:n_pages].copy()

if page_keys and len(page_keys) >= n_pages:
    meta_df["page_key"] = page_keys[:n_pages]
elif "page_key" not in meta_df.columns:
    meta_df["page_key"] = [f"page_{i}" for i in range(n_pages)]

if "join_doc_name" not in meta_df.columns:
    meta_df["join_doc_name"] = "unknown_doc"
if "safe_page" not in meta_df.columns:
    meta_df["safe_page"] = np.arange(n_pages, dtype=np.int32)
if "domain" not in meta_df.columns:
    meta_df["domain"] = "unknown"

meta_df["embed_idx"] = np.arange(n_pages, dtype=np.int32)
embedded_rows = meta_df.copy()
avail_docs = set(embedded_rows["join_doc_name"].astype(str).tolist())
doc_page_lookup = {doc: grp for doc, grp in embedded_rows.groupby("join_doc_name")}

print(f"Loaded encoded pages: {n_pages}")
print(f"Embedding shape sample: {all_page_embeddings[0].shape}")
print(f"Embedding dtype sample: {all_page_embeddings[0].dtype}")
print(f"Metadata columns: {list(embedded_rows.columns)}")
print(f"Docs in index: {len(avail_docs)}")

In [ ]:
# ColPali & ColPaliProcessor — defined from scratch (no colpali_engine import)
# Sources provided by user.
# ==============================================================================

import importlib
from abc import ABC, abstractmethod
from typing import ClassVar, List, Optional, Tuple, Union

import torch
from torch import nn
from PIL import Image
from transformers import (
    AutoImageProcessor,
    AutoTokenizer,
    BatchEncoding,
    BatchFeature,
    PaliGemmaProcessor,
)
from transformers.models.paligemma.modeling_paligemma import (
    PaliGemmaConfig,
    PaliGemmaForConditionalGeneration,
    PaliGemmaPreTrainedModel,
)


# -- Minimal device helper (replaces colpali_engine.utils.get_torch_device) ----
def _get_torch_device(device: str = "auto") -> torch.device:
    if device == "auto":
        return torch.device("cuda" if torch.cuda.is_available() else "cpu")
    return torch.device(device)


# ==============================================================================
# BaseVisualRetrieverProcessor
# ==============================================================================

class BaseVisualRetrieverProcessor(ABC):
    """Base class for visual retriever processors."""

    query_prefix: ClassVar[str] = ""

    @abstractmethod
    def process_images(self, images: List[Image.Image]) -> Union[BatchFeature, BatchEncoding]:
        pass

    @abstractmethod
    def process_texts(self, texts: List[str]) -> Union[BatchFeature, BatchEncoding]:
        pass

    def process_queries(
        self,
        texts: Optional[List[str]] = None,
        queries: Optional[List[str]] = None,
        max_length: int = 50,
        contexts: Optional[List[str]] = None,
        suffix: Optional[str] = None,
    ) -> Union[BatchFeature, BatchEncoding]:
        if texts and queries:
            raise ValueError("Only one of 'texts' or 'queries' should be provided.")
        if queries is not None:
            texts = queries
        elif texts is None:
            raise ValueError("No texts or queries provided.")

        if suffix is None:
            suffix = self.query_augmentation_token * 10

        texts = [self.query_prefix + text + suffix for text in texts]
        return self.process_texts(texts=texts)

    @abstractmethod
    def score(
        self,
        qs: Union[torch.Tensor, List[torch.Tensor]],
        ps: Union[torch.Tensor, List[torch.Tensor]],
        device: Optional[Union[str, torch.device]] = None,
        **kwargs,
    ) -> torch.Tensor:
        pass

    @staticmethod
    def score_single_vector(
        qs: Union[torch.Tensor, List[torch.Tensor]],
        ps: Union[torch.Tensor, List[torch.Tensor]],
        device: Optional[Union[str, torch.device]] = None,
    ) -> torch.Tensor:
        device = device or _get_torch_device("auto")
        if isinstance(qs, list):
            qs = torch.stack(qs).to(device)
        else:
            qs = qs.to(device)
        if isinstance(ps, list):
            ps = torch.stack(ps).to(device)
        else:
            ps = ps.to(device)
        scores = torch.einsum("bd,cd->bc", qs, ps).to(torch.float32)
        return scores

    @staticmethod
    def score_multi_vector(
        qs: Union[torch.Tensor, List[torch.Tensor]],
        ps: Union[torch.Tensor, List[torch.Tensor]],
        batch_size: int = 128,
        device: Optional[Union[str, torch.device]] = None,
    ) -> torch.Tensor:
        device = device or _get_torch_device("auto")
        if len(qs) == 0:
            raise ValueError("No queries provided")
        if len(ps) == 0:
            raise ValueError("No passages provided")

        scores_list: List[torch.Tensor] = []
        for i in range(0, len(qs), batch_size):
            qs_batch = torch.nn.utils.rnn.pad_sequence(
                qs[i : i + batch_size], batch_first=True, padding_value=0
            ).to(device)
            scores_batch = []
            for j in range(0, len(ps), batch_size):
                ps_batch = torch.nn.utils.rnn.pad_sequence(
                    ps[j : j + batch_size], batch_first=True, padding_value=0
                ).to(device)
                scores_batch.append(
                    torch.einsum("bnd,csd->bcns", qs_batch, ps_batch).max(dim=3)[0].sum(dim=2)
                )
            scores_list.append(torch.cat(scores_batch, dim=1).cpu())

        return torch.cat(scores_list, dim=0).to(torch.float32)

    @abstractmethod
    def get_n_patches(
        self,
        image_size: Tuple[int, int],
        *args,
        **kwargs,
    ) -> Tuple[int, int]:
        pass


# ==============================================================================
# ColPali
# ==============================================================================

class ColPali(PaliGemmaPreTrainedModel):
    """
    ColPali model — "ColPali: Efficient Document Retrieval with Vision Language Models".
    """

    main_input_name: ClassVar[str] = "doc_input_ids"
    _keys_to_ignore_on_load_missing = [r"model\.lm_head\.weight"]
    _checkpoint_conversion_mapping = {
        "^model.language_model.model": "model.model.language_model",
        "^model.vision_tower": "model.model.vision_tower",
        "^model.multi_modal_projector": "model.model.multi_modal_projector",
        "^model.language_model.lm_head": "model.lm_head",
        r"^base_model\.model\.custom_text_proj": "custom_text_proj",
    }

    @classmethod
    def from_pretrained(cls, *args, **kwargs):
        key_mapping = kwargs.pop("key_mapping", None)
        if key_mapping is None:
            key_mapping = cls._checkpoint_conversion_mapping
        return super().from_pretrained(*args, **kwargs, key_mapping=key_mapping)

    def __init__(self, config: PaliGemmaConfig, mask_non_image_embeddings: bool = False):
        super().__init__(config=config)

        model = PaliGemmaForConditionalGeneration(config=config)
        if model.model.language_model._tied_weights_keys is not None:
            self._tied_weights_keys = [
                f"model.model.language_model.{k}"
                for k in model.model.language_model._tied_weights_keys
            ]
        self.model = model

        self.dim = 128
        self.custom_text_proj = nn.Linear(
            self.model.config.text_config.hidden_size, self.dim
        )
        self.mask_non_image_embeddings = mask_non_image_embeddings
        self.post_init()

    def forward(self, *args, **kwargs) -> torch.Tensor:
        kwargs.pop("output_hidden_states", None)
        if "pixel_values" in kwargs:
            kwargs["pixel_values"] = kwargs["pixel_values"].to(dtype=self.dtype)

        outputs = self.model(*args, output_hidden_states=True, **kwargs)
        last_hidden_states = outputs.hidden_states[-1]
        proj = self.custom_text_proj(last_hidden_states)
        proj = proj / proj.norm(dim=-1, keepdim=True)
        proj = proj * kwargs["attention_mask"].unsqueeze(-1)

        if "pixel_values" in kwargs and self.mask_non_image_embeddings:
            image_mask = (kwargs["input_ids"] == self.config.image_token_index).unsqueeze(-1)
            proj = proj * image_mask
        return proj

    def get_input_embeddings(self):
        return self.model.model.language_model.get_input_embeddings()

    def set_input_embeddings(self, value):
        self.model.model.language_model.set_input_embeddings(value)

    def get_output_embeddings(self):
        return self.model.model.language_model.get_output_embeddings()

    def set_output_embeddings(self, new_embeddings):
        self.model.model.language_model.set_output_embeddings(new_embeddings)

    def set_decoder(self, decoder):
        self.model.model.language_model.set_decoder(decoder)

    def get_decoder(self):
        return self.model.model.language_model.get_decoder()

    def tie_weights(self, *args, **kwargs):
        return self.model.model.language_model.tie_weights(*args, **kwargs)

    def resize_token_embeddings(
        self,
        new_num_tokens: Optional[int] = None,
        pad_to_multiple_of=None,
    ) -> nn.Embedding:
        model_embeds = self.model.model.language_model.resize_token_embeddings(
            new_num_tokens, pad_to_multiple_of
        )
        self.config.text_config.vocab_size = model_embeds.num_embeddings
        self.config.vocab_size = model_embeds.num_embeddings
        self.model.vocab_size = model_embeds.num_embeddings
        return model_embeds

    @property
    def patch_size(self) -> int:
        return self.model.vision_tower.config.patch_size


# ==============================================================================
# ColPaliProcessor
# ==============================================================================

class ColPaliProcessor(BaseVisualRetrieverProcessor, PaliGemmaProcessor):
    """Processor for ColPali."""

    visual_prompt_prefix: ClassVar[str] = "<image><bos>Describe the image."

    def __init__(self, image_processor=None, tokenizer=None, chat_template=None, **kwargs):
        super().__init__(
            image_processor=image_processor,
            tokenizer=tokenizer,
            chat_template=chat_template,
            **kwargs,
        )

    @classmethod
    def from_pretrained(cls, *args, **kwargs):
        try:
            instance = super().from_pretrained(*args, **kwargs)
            if (getattr(instance, "image_processor", None) is None
                    or getattr(instance, "tokenizer", None) is None):
                raise ValueError("Loaded processor is missing image_processor or tokenizer.")
            return instance
        except ValueError as exc:
            msg = str(exc)
            if "image_seq_length" not in msg and "missing image_processor" not in msg:
                raise

        load_kwargs = {
            key: kwargs[key]
            for key in ("cache_dir", "force_download", "local_files_only",
                        "revision", "token", "trust_remote_code")
            if key in kwargs
        }
        image_processor = AutoImageProcessor.from_pretrained(*args, **load_kwargs)
        tokenizer = AutoTokenizer.from_pretrained(*args, **load_kwargs)
        return cls(image_processor=image_processor, tokenizer=tokenizer)

    @property
    def query_augmentation_token(self) -> str:
        return self.tokenizer.pad_token

    def process_images(self, images: List[Image.Image]) -> Union[BatchFeature, BatchEncoding]:
        images = [image.convert("RGB") for image in images]
        return self(
            text=[self.visual_prompt_prefix] * len(images),
            images=images,
            return_tensors="pt",
            padding="longest",
        )

    def process_texts(self, texts: List[str]) -> Union[BatchFeature, BatchEncoding]:
        return self.tokenizer(
            [self.tokenizer.bos_token + text for text in texts],
            text_pair=None,
            return_token_type_ids=True,
            return_tensors="pt",
            padding="longest",
            truncation=True,
            max_length=512,
        )

    def score(
        self,
        qs: List[torch.Tensor],
        ps: List[torch.Tensor],
        device: Optional[Union[str, torch.device]] = None,
        **kwargs,
    ) -> torch.Tensor:
        return self.score_multi_vector(qs, ps, device=device, **kwargs)

    def get_n_patches(
        self,
        image_size: Tuple[int, int],
        patch_size: int,
    ) -> Tuple[int, int]:
        n_patches_x = self.image_processor.size["width"] // patch_size
        n_patches_y = self.image_processor.size["height"] // patch_size
        return n_patches_x, n_patches_y

    def get_image_mask(self, batch_images: BatchFeature) -> torch.Tensor:
        return batch_images.input_ids == self.image_token_id


print("✅ ColPali and ColPaliProcessor class definitions ready.")

In [ ]:
# ==============================================================================
# Load model & processor from local checkpoint
# ==============================================================================
from peft import PeftModel
gc.collect()
torch.cuda.empty_cache()

# from colpali_engine.models import ColPali, ColPaliProcessor

print(">>> Loading ColPali model...")
query_model = ColPali.from_pretrained(
    COLPALI_BASE,
    torch_dtype=torch.bfloat16,
    device_map="cuda",
    attn_implementation="eager",   # required to expose attention weights
                                    # for Methods 2 & 4; no impact on 1/3/5/6
)

query_model = PeftModel.from_pretrained(
    query_model,
    COLPALI_LORA
)
print(">>> Loading ColPaliProcessor...")
query_processor = ColPaliProcessor.from_pretrained(COLPALI_LORA)

print("✅ ColPali ready")
print(f"   proj dim : {query_model.dim}")

In [ ]:
# ==============================================================================
# SAFE/FAST PRESET — re-encode control panel
# Run this cell BEFORE the re-encode cell.
# ==============================================================================

import shutil

# Choose profile: "fast" (higher throughput) or "safe" (max stability)
ENCODE_PROFILE = "fast"

if ENCODE_PROFILE == "fast":
    PERF_CFG = dict(globals().get("PERF_CFG", {}))
    PERF_CFG.update({
        "BATCH_SIZE_ENCODE": 8,    # faster than conservative mode
        "SAVE_EVERY": 0,           # keep disabled to avoid checkpoint spikes
        "SHARD_SIZE": 1024,        # larger shard reduces disk I/O overhead
        "PARQUET_BATCH_ROWS": 64,  # faster parquet stream
        "CLEANUP_EVERY_ROWS": 512, # less frequent cleanup for speed
        "MAX_ROWS_PER_RUN": 0,     # 0 = full continuous run
    })
else:
    PERF_CFG = dict(globals().get("PERF_CFG", {}))
    PERF_CFG.update({
        "BATCH_SIZE_ENCODE": 4,
        "SAVE_EVERY": 0,
        "SHARD_SIZE": 256,
        "PARQUET_BATCH_ROWS": 16,
        "CLEANUP_EVERY_ROWS": 128,
        "MAX_ROWS_PER_RUN": 0,
    })

# Keep bf16 for speed/stability balance
AUTOCAST_DTYPE = torch.bfloat16

# Prefer stability over max throughput
if torch.cuda.is_available():
    torch.backends.cudnn.benchmark = False
    torch.cuda.empty_cache()

# Set True to start completely fresh from page 0
RESET_REENCODE_STATE = True

if RESET_REENCODE_STATE:
    safe_ckpt = os.path.join(WORKING_DIR, "vidore_v3_reencoded_colpali.ckpt.pkl")
    safe_manifest = os.path.join(WORKING_DIR, "vidore_v3_reencoded_colpali.pkl")
    safe_shard_dir = os.path.join(WORKING_DIR, "vidore_v3_reencoded_colpali_shards")

    if os.path.isfile(safe_ckpt):
        try:
            os.remove(safe_ckpt)
            print(f">>> Removed checkpoint: {safe_ckpt}")
        except Exception as e:
            print(f">>> Could not remove checkpoint: {e}")

    if os.path.isfile(safe_manifest):
        try:
            os.remove(safe_manifest)
            print(f">>> Removed manifest: {safe_manifest}")
        except Exception as e:
            print(f">>> Could not remove manifest: {e}")

    if os.path.isdir(safe_shard_dir):
        try:
            shutil.rmtree(safe_shard_dir, ignore_errors=True)
            print(f">>> Removed shard directory: {safe_shard_dir}")
        except Exception as e:
            print(f">>> Could not remove shard directory: {e}")

gc.collect()

print(f">>> PRESET loaded (profile={ENCODE_PROFILE})")
print(f"BATCH_SIZE_ENCODE = {PERF_CFG['BATCH_SIZE_ENCODE']}")
print(f"SAVE_EVERY        = {PERF_CFG['SAVE_EVERY']} (mid-run checkpoint disabled)")
print(f"SHARD_SIZE        = {PERF_CFG['SHARD_SIZE']}")
print(f"PARQUET_BATCH_ROWS= {PERF_CFG['PARQUET_BATCH_ROWS']}")
print(f"CLEANUP_EVERY_ROWS= {PERF_CFG['CLEANUP_EVERY_ROWS']}")
print(f"MAX_ROWS_PER_RUN  = {PERF_CFG['MAX_ROWS_PER_RUN']} (0 means continuous full run)")
print(f"AUTOCAST_DTYPE    = {AUTOCAST_DTYPE}")
print(f"RESET_REENCODE_STATE = {RESET_REENCODE_STATE}")
print("Now run the re-encode cell.")

In [ ]:
# ==============================================================================
# Re-encode ViDoRe corpus -> ColPali index (RAM-safe sharded mode, streamed IO)
# Uses slice-style runs like the reference notebook: process a safe chunk, save, rerun.
# ==============================================================================

from pathlib import Path
import io
import pyarrow as pa
import pyarrow.parquet as pq

print(">>> Re-encoding ViDoRe corpus with current ColPali model (RAM-safe sharded mode)...")
device = "cuda" if torch.cuda.is_available() else "cpu"

# ---- Re-encode config ----
REENCODE_OUTPUT_PKL = os.path.join(WORKING_DIR, "vidore_v3_reencoded_colpali.pkl")
REENCODE_CHECKPOINT_PKL = os.path.join(WORKING_DIR, "vidore_v3_reencoded_colpali.ckpt.pkl")
REENCODE_SHARD_DIR = os.path.join(WORKING_DIR, "vidore_v3_reencoded_colpali_shards")
REENCODE_DOMAIN_FILTER = DOMAIN_FILTER if DOMAIN_FILTER else []

# Prefer PERF_CFG if available
_default_bs = int(globals().get("PERF_CFG", {}).get("BATCH_SIZE_ENCODE", 4))
REENCODE_BATCH_SIZE = int(max(2, _default_bs))
REENCODE_SAVE_EVERY = int(globals().get("PERF_CFG", {}).get("SAVE_EVERY", 0))

REENCODE_SHARD_SIZE = int(globals().get("PERF_CFG", {}).get("SHARD_SIZE", 256))
REENCODE_PARQUET_BATCH_ROWS = int(globals().get("PERF_CFG", {}).get("PARQUET_BATCH_ROWS", 16))
REENCODE_CLEANUP_EVERY_ROWS = int(globals().get("PERF_CFG", {}).get("CLEANUP_EVERY_ROWS", 128))

# Critical: process only a bounded chunk per run; rerun cell to continue.
REENCODE_MAX_ROWS_PER_RUN = int(globals().get("PERF_CFG", {}).get("MAX_ROWS_PER_RUN", 2200))

REENCODE_MIN_BATCH = 1
REENCODE_MAX_PAGES = None
REENCODE_MAX_PAGES_PER_DOMAIN = None
REENCODE_SHOW_TOTAL = True
CORPUS_COLUMNS = ["corpus_id", "image", "doc_id", "page_number_in_doc"]
EMBED_STORAGE_DTYPE = np.float16

root = Path(VIDORE_DATASET_ROOT)
if not root.exists():
    raise FileNotFoundError(f"ViDoRe dataset root not found: {VIDORE_DATASET_ROOT}")
os.makedirs(REENCODE_SHARD_DIR, exist_ok=True)

domain_dirs = [p for p in sorted(root.iterdir()) if p.is_dir()]
if REENCODE_DOMAIN_FILTER:
    domain_dirs = [p for p in domain_dirs if p.name in set(REENCODE_DOMAIN_FILTER)]

print(f"Domains to encode ({len(domain_dirs)}): {[d.name for d in domain_dirs]}")
print(
    f"Batch={REENCODE_BATCH_SIZE} | ShardSize={REENCODE_SHARD_SIZE} | "
    f"SaveEvery={REENCODE_SAVE_EVERY} | ParquetBatch={REENCODE_PARQUET_BATCH_ROWS} | "
    f"CleanupEvery={REENCODE_CLEANUP_EVERY_ROWS} | MaxRowsPerRun={REENCODE_MAX_ROWS_PER_RUN}"
)

if torch.cuda.is_available():
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    torch.backends.cudnn.benchmark = False
    torch.set_float32_matmul_precision("high")


def _norm_int_string(v):
    if pd.isna(v):
        return ""
    s = str(v).strip()
    if not s:
        return ""
    try:
        f = float(s)
        if f.is_integer():
            return str(int(f))
    except Exception:
        pass
    return s


def _as_pil_image(v):
    if isinstance(v, Image.Image):
        return v.convert("RGB")

    if isinstance(v, dict):
        if v.get("bytes") is not None:
            return Image.open(io.BytesIO(v["bytes"])).convert("RGB")
        if v.get("path"):
            return Image.open(v["path"]).convert("RGB")

    if isinstance(v, (bytes, bytearray)):
        return Image.open(io.BytesIO(v)).convert("RGB")

    if isinstance(v, np.ndarray):
        arr = v
        if arr.dtype != np.uint8:
            arr = np.clip(arr, 0, 255).astype(np.uint8)
        return Image.fromarray(arr).convert("RGB")

    raise ValueError(f"Unsupported image type: {type(v)}")


def _arrow_scalar_to_py(arr, idx):
    try:
        return arr[idx].as_py()
    except Exception:
        return None


def _release_arrow_memory():
    try:
        pa.default_memory_pool().release_unused()
    except Exception:
        pass


class _NullCtx:
    def __enter__(self):
        return None

    def __exit__(self, exc_type, exc, tb):
        return False


AUTOCAST_DTYPE = globals().get("AUTOCAST_DTYPE", torch.bfloat16)

all_page_embeddings = []
page_keys = []
meta_records = []
shard_files = []
next_shard_id = 0

total_rows_seen = 0
total_rows_encoded = 0
total_rows_failed = 0

batch_images = []
batch_meta = []
current_batch_size = REENCODE_BATCH_SIZE
domain_seen_counter = {}
cleanup_tick = 0


def _make_shard_path(shard_id):
    return os.path.join(REENCODE_SHARD_DIR, f"shard_{shard_id:05d}.pkl")


def _spill_to_shard(force=False):
    global all_page_embeddings
    global page_keys
    global meta_records
    global shard_files
    global next_shard_id

    if len(all_page_embeddings) == 0:
        return
    if (not force) and len(all_page_embeddings) < REENCODE_SHARD_SIZE:
        return

    shard_path = _make_shard_path(next_shard_id)
    shard_payload = {
        "fused_index": all_page_embeddings,
        "page_keys": page_keys,
        "metadata": meta_records,
    }
    with open(shard_path, "wb") as sf:
        pickle.dump(shard_payload, sf, protocol=pickle.HIGHEST_PROTOCOL)

    shard_files.append(shard_path)
    next_shard_id += 1

    n_saved = len(all_page_embeddings)
    all_page_embeddings = []
    page_keys = []
    meta_records = []

    gc.collect()
    _release_arrow_memory()

    print(f">>> Shard saved: {os.path.basename(shard_path)} | pages={n_saved} | total_encoded={total_rows_encoded}")


def _save_checkpoint():
    ckpt_payload = {
        "format": "vidore_sharded_v1",
        "shard_files": shard_files,
        "next_shard_id": next_shard_id,
        "active_fused_index": all_page_embeddings,
        "active_page_keys": page_keys,
        "active_metadata": meta_records,
        "total_rows_seen": total_rows_seen,
        "total_rows_encoded": total_rows_encoded,
        "total_rows_failed": total_rows_failed,
        "current_batch_size": current_batch_size,
        "domain_seen_counter": domain_seen_counter,
    }
    with open(REENCODE_CHECKPOINT_PKL, "wb") as f:
        pickle.dump(ckpt_payload, f, protocol=pickle.HIGHEST_PROTOCOL)


def _encode_batch(batch_imgs, batch_metas, run_device):
    if run_device == "cuda":
        ctx = torch.autocast(device_type="cuda", dtype=AUTOCAST_DTYPE, enabled=True)
    else:
        ctx = _NullCtx()

    with ctx:
        inputs = query_processor.process_images(batch_imgs).to(run_device)
        if "token_type_ids" not in inputs and "input_ids" in inputs:
            inputs["token_type_ids"] = torch.zeros_like(inputs["input_ids"])

        proj = query_model(**inputs)

        if hasattr(proj, "last_hidden_state") and proj.last_hidden_state is not None:
            proj = proj.last_hidden_state
        elif isinstance(proj, (tuple, list)) and len(proj) > 0:
            proj = proj[0]

        attn_mask = inputs["attention_mask"]
        local_embeddings = []
        for i in range(proj.shape[0]):
            valid_idx = torch.where(attn_mask[i] > 0)[0]
            emb = proj[i][valid_idx].detach().cpu().numpy().astype(EMBED_STORAGE_DTYPE, copy=False)
            local_embeddings.append(emb)

    for emb, meta in zip(local_embeddings, batch_metas):
        all_page_embeddings.append(emb)
        page_keys.append(meta["page_key"])
        meta_records.append(meta)

    del inputs
    del proj
    del attn_mask
    del local_embeddings


def _close_images(images):
    for im in images:
        try:
            if hasattr(im, "close"):
                im.close()
        except Exception:
            pass


def _flush_batch_oom_safe(force_batch_size=None):
    global total_rows_encoded
    global total_rows_failed
    global current_batch_size

    if not batch_images:
        return

    run_bs = min(force_batch_size or current_batch_size, len(batch_images))

    while run_bs >= REENCODE_MIN_BATCH:
        try:
            with torch.inference_mode():
                _encode_batch(batch_images[:run_bs], batch_meta[:run_bs], device)

            _close_images(batch_images[:run_bs])
            del batch_images[:run_bs]
            del batch_meta[:run_bs]
            total_rows_encoded += run_bs

            current_batch_size = max(REENCODE_MIN_BATCH, min(current_batch_size, run_bs))
            _spill_to_shard(force=False)
            return

        except RuntimeError as e:
            msg = str(e).lower()
            is_oom = "out of memory" in msg or "cuda out of memory" in msg
            if not is_oom:
                raise

            torch.cuda.empty_cache()
            run_bs = run_bs // 2
            current_batch_size = max(REENCODE_MIN_BATCH, run_bs)

    _close_images(batch_images[:1])
    _ = batch_images.pop(0)
    _ = batch_meta.pop(0)
    total_rows_failed += 1


def _load_resume_checkpoint_if_any():
    global all_page_embeddings
    global page_keys
    global meta_records
    global shard_files
    global next_shard_id
    global total_rows_seen
    global total_rows_encoded
    global total_rows_failed
    global current_batch_size
    global domain_seen_counter

    if not os.path.isfile(REENCODE_CHECKPOINT_PKL):
        return

    try:
        with open(REENCODE_CHECKPOINT_PKL, "rb") as f:
            ckpt = pickle.load(f)

        if not isinstance(ckpt, dict):
            return

        shard_files = ckpt.get("shard_files", [])
        next_shard_id = int(ckpt.get("next_shard_id", len(shard_files)))
        all_page_embeddings = ckpt.get("active_fused_index", [])
        page_keys = ckpt.get("active_page_keys", [])
        meta_records = ckpt.get("active_metadata", [])

        total_rows_seen = int(ckpt.get("total_rows_seen", 0))
        total_rows_encoded = int(ckpt.get("total_rows_encoded", 0))
        total_rows_failed = int(ckpt.get("total_rows_failed", 0))
        current_batch_size = int(max(REENCODE_MIN_BATCH, ckpt.get("current_batch_size", current_batch_size)))
        domain_seen_counter = dict(ckpt.get("domain_seen_counter", {}))

        print(
            f">>> Resumed checkpoint: seen={total_rows_seen}, encoded={total_rows_encoded}, "
            f"shards={len(shard_files)}, active_buf={len(all_page_embeddings)}"
        )
    except Exception as e:
        print(f">>> Checkpoint load failed, starting fresh: {e}")


_load_resume_checkpoint_if_any()
start_rows_seen_run = int(total_rows_seen)
run_rows_done = 0
run_reached_limit = False

corpus_files = []
for domain_dir in domain_dirs:
    for fp in sorted(domain_dir.rglob("*.parquet")):
        pstr = str(fp).replace("\\", "/").lower()
        if "/corpus/" in pstr:
            corpus_files.append((domain_dir.name, fp))

if not corpus_files:
    raise ValueError("No corpus parquet files found under ViDoRe root.")

print(f"Corpus parquet files found: {len(corpus_files)}")

estimated_total = None
if REENCODE_SHOW_TOTAL:
    try:
        per_domain_count = dict(domain_seen_counter)
        acc = 0
        for domain_name, fp in corpus_files:
            n_rows = int(pq.ParquetFile(str(fp)).metadata.num_rows)

            if REENCODE_MAX_PAGES_PER_DOMAIN is not None:
                used = int(per_domain_count.get(domain_name, 0))
                remain = max(0, REENCODE_MAX_PAGES_PER_DOMAIN - used)
                n_rows = min(n_rows, remain)
                per_domain_count[domain_name] = used + n_rows

            if REENCODE_MAX_PAGES is not None:
                remain_all = max(0, REENCODE_MAX_PAGES - acc)
                if remain_all <= 0:
                    break
                n_rows = min(n_rows, remain_all)

            acc += n_rows

            if REENCODE_MAX_PAGES is not None and acc >= REENCODE_MAX_PAGES:
                break

        estimated_total = acc
    except Exception:
        estimated_total = None

if estimated_total is not None:
    print(f"Estimated corpus pages to encode: {estimated_total}")
    pbar = tqdm(total=estimated_total, desc="Re-encoding corpus pages")
    if total_rows_seen > 0:
        pbar.update(min(total_rows_seen, estimated_total))
else:
    print("Estimated corpus pages: unavailable (fallback to open-ended progress bar)")
    pbar = tqdm(total=None, desc="Re-encoding corpus pages")
    if total_rows_seen > 0:
        pbar.update(total_rows_seen)

rows_to_skip = max(0, total_rows_seen)
stop_all = False

for domain_name, fp in corpus_files:
    if stop_all:
        break

    domain_seen = int(domain_seen_counter.get(domain_name, 0))
    if REENCODE_MAX_PAGES_PER_DOMAIN is not None and domain_seen >= REENCODE_MAX_PAGES_PER_DOMAIN:
        continue

    try:
        parquet_file = pq.ParquetFile(str(fp))
        record_batches = parquet_file.iter_batches(
            batch_size=REENCODE_PARQUET_BATCH_ROWS,
            columns=CORPUS_COLUMNS,
            use_threads=False,
        )
    except Exception:
        continue

    for rb in record_batches:
        n_rb = rb.num_rows
        idx_cid = rb.schema.get_field_index("corpus_id")
        idx_img = rb.schema.get_field_index("image")
        idx_doc = rb.schema.get_field_index("doc_id")
        idx_pg = rb.schema.get_field_index("page_number_in_doc")

        col_cid = rb.column(idx_cid) if idx_cid >= 0 else None
        col_img = rb.column(idx_img) if idx_img >= 0 else None
        col_doc = rb.column(idx_doc) if idx_doc >= 0 else None
        col_pg = rb.column(idx_pg) if idx_pg >= 0 else None

        for i in range(n_rb):
            if rows_to_skip > 0:
                rows_to_skip -= 1
                continue

            if REENCODE_MAX_PAGES is not None and total_rows_seen >= REENCODE_MAX_PAGES:
                stop_all = True
                break

            if REENCODE_MAX_PAGES_PER_DOMAIN is not None and domain_seen >= REENCODE_MAX_PAGES_PER_DOMAIN:
                break

            if REENCODE_MAX_ROWS_PER_RUN > 0 and run_rows_done >= REENCODE_MAX_ROWS_PER_RUN:
                run_reached_limit = True
                stop_all = True
                break

            total_rows_seen += 1
            run_rows_done += 1
            cleanup_tick += 1

            try:
                img_raw = _arrow_scalar_to_py(col_img, i) if col_img is not None else None
                img = _as_pil_image(img_raw)
                cid = _norm_int_string(_arrow_scalar_to_py(col_cid, i) if col_cid is not None else None)

                doc_raw = _arrow_scalar_to_py(col_doc, i) if col_doc is not None else None
                doc_id = "" if pd.isna(doc_raw) else str(doc_raw).strip()

                page_num = _arrow_scalar_to_py(col_pg, i) if col_pg is not None else np.nan

                if not cid or not doc_id:
                    try:
                        img.close()
                    except Exception:
                        pass
                    total_rows_failed += 1
                    pbar.update(1)
                    continue

                try:
                    safe_page = int(float(page_num)) if pd.notna(page_num) else -1
                except Exception:
                    safe_page = -1

                meta = {
                    "corpus_id": cid,
                    "doc_id": doc_id,
                    "join_doc_name": doc_id,
                    "page_number_in_doc": safe_page,
                    "safe_page": safe_page,
                    "domain": domain_name,
                    "source_parquet": str(fp),
                }

                page_key = f"{domain_name}/{doc_id}#p{safe_page}#cid{cid}"
                meta["page_key"] = page_key

                batch_images.append(img)
                batch_meta.append(meta)
                domain_seen += 1
                domain_seen_counter[domain_name] = domain_seen

                if len(batch_images) >= current_batch_size:
                    _flush_batch_oom_safe()

            except Exception:
                total_rows_failed += 1

            pbar.update(1)

            if REENCODE_SAVE_EVERY > 0 and total_rows_seen > 0 and (total_rows_seen % REENCODE_SAVE_EVERY == 0):
                _save_checkpoint()

            if cleanup_tick >= REENCODE_CLEANUP_EVERY_ROWS:
                cleanup_tick = 0
                gc.collect()
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()
                _release_arrow_memory()

        del col_cid, col_img, col_doc, col_pg
        del rb
        gc.collect()
        _release_arrow_memory()

        if stop_all:
            break

    del parquet_file
    gc.collect()
    _release_arrow_memory()

while batch_images:
    _flush_batch_oom_safe(force_batch_size=len(batch_images))

_spill_to_shard(force=True)
pbar.close()

# Save checkpoint on every run boundary (important for slice-style continuation).
_save_checkpoint()

if len(shard_files) == 0 and len(all_page_embeddings) == 0:
    raise ValueError("No pages were encoded. Check corpus parquet schema and image decoding.")

# Write/refresh manifest every run so current partial index can be inspected.
manifest_payload = {
    "format": "vidore_sharded_v1",
    "storage_dtype": str(EMBED_STORAGE_DTYPE),
    "shard_files": shard_files,
    "num_pages": int(total_rows_encoded),
}

with open(REENCODE_OUTPUT_PKL, "wb") as f:
    pickle.dump(manifest_payload, f, protocol=pickle.HIGHEST_PROTOCOL)

# If full pass finished (did not stop by per-run limit and no global limit left), clear checkpoint.
is_completed = (not run_reached_limit) and ((REENCODE_MAX_PAGES is None) or (total_rows_seen >= REENCODE_MAX_PAGES) or (rows_to_skip == 0 and not stop_all))
if is_completed and os.path.isfile(REENCODE_CHECKPOINT_PKL):
    try:
        os.remove(REENCODE_CHECKPOINT_PKL)
    except Exception:
        pass

print(f"\nRows this run                : {run_rows_done}")
print(f"Rows seen / encoded / failed : {total_rows_seen} / {total_rows_encoded} / {total_rows_failed}")
print(f"Batch size (stable)          : {current_batch_size}")
print(f"Shard files                  : {len(shard_files)}")
print(f"Saved manifest PKL           : {REENCODE_OUTPUT_PKL}")

INDEX_PKL_PATH = REENCODE_OUTPUT_PKL
print(f"INDEX_PKL_PATH updated to: {INDEX_PKL_PATH}")

# Build metadata-only in-memory table from existing shards
meta_list = []
for sp in shard_files:
    with open(sp, "rb") as sf:
        sh = pickle.load(sf)
    meta_list.extend(sh.get("metadata", []))

meta_df = pd.DataFrame(meta_list)
meta_df["embed_idx"] = np.arange(len(meta_df), dtype=np.int32)
embedded_rows = meta_df.copy()
avail_docs = set(embedded_rows["join_doc_name"].astype(str).tolist())
doc_page_lookup = {doc: grp for doc, grp in embedded_rows.groupby("join_doc_name")}

print(f"In-memory metadata rows ready: {len(embedded_rows)}")
print(f"Metadata columns             : {list(embedded_rows.columns)}")
print(f"Docs in index                : {len(avail_docs)}")

if run_reached_limit:
    print("\nRun reached MAX_ROWS_PER_RUN safely.")
    print("Rerun this same cell to continue from checkpoint (no need to reset kernel).")
else:
    print("\nRe-encode pass finished for current limits.")
    print("Next: run Cell 3 (load encoded index) to materialize embeddings when needed.")

In [ ]:
# DEBUG CELL — run this before QA mapping cell (ViDoRe)
from pathlib import Path

root = Path(VIDORE_DATASET_ROOT)
if not root.exists():
    raise FileNotFoundError(f"ViDoRe dataset root not found: {VIDORE_DATASET_ROOT}")

domain_dirs = [p for p in sorted(root.iterdir()) if p.is_dir()]
if DOMAIN_FILTER:
    domain_dirs = [p for p in domain_dirs if p.name in set(DOMAIN_FILTER)]

print(f"ViDoRe root: {root}")
print(f"Domains selected ({len(domain_dirs)}): {[p.name for p in domain_dirs]}")
print(f"Indexed pages: {len(embedded_rows)}")

if "domain" in embedded_rows.columns:
    print("\nIndexed pages per domain:")
    print(embedded_rows.groupby("domain").size().sort_values(ascending=False).head(20))

query_parquets = []
for d in domain_dirs:
    for fp in d.rglob("*.parquet"):
        pstr = str(fp).replace("\\", "/").lower()
        if "/corpus/" in pstr:
            continue
        query_parquets.append(fp)

print(f"\nNon-corpus parquet files found: {len(query_parquets)}")
for fp in query_parquets[:20]:
    print(" -", fp)

if query_parquets:
    sample_df = pd.read_parquet(query_parquets[0])
    print("\nSample query parquet:", query_parquets[0])
    print("Columns:", list(sample_df.columns))
    print(sample_df.head(2).to_string())

In [ ]:
# ==============================================================================
# Cell 8 — Build QA Pairs from ViDoRe (schema-driven, strict)
# ==============================================================================

from pathlib import Path
from collections import defaultdict

print("\nBuilding QA pairs from ViDoRe query files (schema-driven)...")

root = Path(VIDORE_DATASET_ROOT)
if not root.exists():
    raise FileNotFoundError(f"ViDoRe dataset root not found: {VIDORE_DATASET_ROOT}")

domain_dirs = [p for p in sorted(root.iterdir()) if p.is_dir()]
if DOMAIN_FILTER:
    domain_dirs = [p for p in domain_dirs if p.name in set(DOMAIN_FILTER)]

# ------------------------------------------------------------------------------
# Build strict corpus_id -> embed_idx mapping from encoded index metadata
# Also build domain-aware mapping to avoid cross-domain corpus_id collisions.
# ------------------------------------------------------------------------------
if "corpus_id" not in embedded_rows.columns:
    raise ValueError(
        "`corpus_id` column is missing in encoded index metadata. "
        "ViDoRe qrels are keyed by corpus_id, so this mapping is required. "
        "Please re-encode index while saving corpus_id in metadata."
    )

if "domain" not in embedded_rows.columns:
    embedded_rows = embedded_rows.copy()
    embedded_rows["domain"] = "unknown"


def _norm_key(v):
    s = "" if pd.isna(v) else str(v).strip()
    if not s:
        return ""
    # Convert numeric-like IDs to canonical int string to align parquet dtypes
    try:
        f = float(s)
        if f.is_integer():
            return str(int(f))
    except Exception:
        pass
    return s


def _norm_domain(v):
    return "" if pd.isna(v) else str(v).strip().lower()


# Global mapping (fallback)
corpus_id_to_embed = defaultdict(list)
# Domain-aware mapping (preferred)
domain_corpus_id_to_embed = defaultdict(list)

for _, r in embedded_rows.iterrows():
    cid_key = _norm_key(r.get("corpus_id"))
    if not cid_key:
        continue

    d_key = _norm_domain(r.get("domain"))
    eidx = int(r["embed_idx"])

    corpus_id_to_embed[cid_key].append(eidx)
    domain_corpus_id_to_embed[(d_key, cid_key)].append(eidx)

if not corpus_id_to_embed:
    raise ValueError("No valid corpus_id found in encoded metadata.")

# Keep deterministic order and no duplicates
for k in list(corpus_id_to_embed.keys()):
    corpus_id_to_embed[k] = sorted(set(corpus_id_to_embed[k]))
for k in list(domain_corpus_id_to_embed.keys()):
    domain_corpus_id_to_embed[k] = sorted(set(domain_corpus_id_to_embed[k]))

# corpus_id collision diagnostics across domains
cid_domains = defaultdict(set)
for (d_key, cid_key), _ in domain_corpus_id_to_embed.items():
    cid_domains[cid_key].add(d_key)
collision_cids = [cid for cid, ds in cid_domains.items() if len(ds) > 1]

# ------------------------------------------------------------------------------
# Build QA pairs from strict queries + qrels tables
# ------------------------------------------------------------------------------
qa_pairs = []

scan_files = 0
used_query_files = 0
used_qrels_files = 0

queries_total = 0
qrels_total = 0
qrels_positive = 0
qrels_mapped = 0

missing_qid_in_queries = 0
queries_without_positive_qrels = 0
queries_with_unmapped_corpus = 0

mapped_with_domain_key = 0
mapped_with_global_fallback = 0

for domain_dir in domain_dirs:
    domain_name = domain_dir.name
    domain_key = _norm_domain(domain_name)

    query_frames = []
    qrels_frames = []

    for fp in sorted(domain_dir.rglob("*.parquet")):
        pstr = str(fp).replace("\\", "/").lower()
        if "/corpus/" in pstr:
            continue

        scan_files += 1
        try:
            df = pd.read_parquet(fp)
        except Exception:
            continue

        if df is None or df.empty:
            continue

        if "/queries/" in pstr:
            if "query_id" in df.columns and "query" in df.columns:
                query_frames.append(df)
                used_query_files += 1
            continue

        if "/qrels/" in pstr:
            if "query_id" in df.columns and "corpus_id" in df.columns:
                qrels_frames.append(df)
                used_qrels_files += 1
            continue

    if not query_frames or not qrels_frames:
        continue

    queries_df = pd.concat(query_frames, ignore_index=True)
    qrels_df = pd.concat(qrels_frames, ignore_index=True)

    queries_total += len(queries_df)
    qrels_total += len(qrels_df)

    # Positive relevance only (ViDoRe: 1/2 are relevant)
    if "score" in qrels_df.columns:
        qrels_pos = qrels_df[pd.to_numeric(qrels_df["score"], errors="coerce") > 0].copy()
    else:
        qrels_pos = qrels_df.copy()
    qrels_positive += len(qrels_pos)

    # query_id -> list(corpus_id)
    qid_to_cids = defaultdict(list)
    for _, r in qrels_pos.iterrows():
        qid_key = _norm_key(r.get("query_id"))
        cid_key = _norm_key(r.get("corpus_id"))
        if qid_key and cid_key:
            qid_to_cids[qid_key].append(cid_key)

    # Build query_id -> GT embed indices
    qid_to_gt = {}

    for qid_key, cids in qid_to_cids.items():
        gt = set()
        mapped_any = False

        for cid_key in cids:
            # Prefer exact (domain, corpus_id)
            dom_pair = (domain_key, cid_key)
            if dom_pair in domain_corpus_id_to_embed:
                mapped_any = True
                gt.update(domain_corpus_id_to_embed[dom_pair])
                mapped_with_domain_key += 1
                continue

            # Fallback global corpus_id
            if cid_key in corpus_id_to_embed:
                mapped_any = True
                gt.update(corpus_id_to_embed[cid_key])
                mapped_with_global_fallback += 1

        if mapped_any and gt:
            qid_to_gt[qid_key] = sorted(gt)
            qrels_mapped += 1

    # Build QA rows from queries
    for _, r in queries_df.iterrows():
        qid_key = _norm_key(r.get("query_id"))
        if not qid_key:
            missing_qid_in_queries += 1
            continue

        qtext = r.get("query")
        question = "" if pd.isna(qtext) else str(qtext).strip()
        if not question:
            continue

        if qid_key not in qid_to_cids:
            queries_without_positive_qrels += 1
            continue

        gt_indices = qid_to_gt.get(qid_key, [])
        if not gt_indices:
            queries_with_unmapped_corpus += 1
            continue

        # Optional doc name from first mapped page metadata
        if "join_doc_name" in embedded_rows.columns and len(gt_indices) > 0:
            doc_name = str(embedded_rows.iloc[int(gt_indices[0])].get("join_doc_name", domain_name))
        else:
            doc_name = domain_name

        qa_pairs.append(
            {
                "question": question,
                "gt_embed_indices": gt_indices,
                "doc_name": doc_name,
                "domain": domain_name,
            }
        )

# ------------------------------------------------------------------------------
# Summary diagnostics
# ------------------------------------------------------------------------------
print(f"Parquet files scanned            : {scan_files}")
print(f"Query files used                : {used_query_files}")
print(f"Qrels files used                : {used_qrels_files}")
print(f"Queries rows total              : {queries_total}")
print(f"Qrels rows total                : {qrels_total}")
print(f"Qrels rows positive             : {qrels_positive}")
print(f"Qrels query_ids mapped to index : {qrels_mapped}")
print(f"QA pairs built                  : {len(qa_pairs)}")
print(f"Queries w/o positive qrels      : {queries_without_positive_qrels}")
print(f"Queries unmapped corpus_id      : {queries_with_unmapped_corpus}")
print(f"Queries missing query_id        : {missing_qid_in_queries}")

print(f"Mapped by (domain, corpus_id)   : {mapped_with_domain_key}")
print(f"Mapped by global fallback       : {mapped_with_global_fallback}")
print(f"Corpus_id cross-domain collisions: {len(collision_cids)}")

cid_sizes = [len(v) for v in corpus_id_to_embed.values()]
if cid_sizes:
    print(f"corpus_id lookup size           : {len(corpus_id_to_embed)}")
    print(
        f"corpus_id pages stats           : min={min(cid_sizes)} | "
        f"p50={int(np.median(cid_sizes))} | max={max(cid_sizes)}"
    )

if not qa_pairs:
    raise ValueError(
        "No QA pairs built from strict query_id/corpus_id mapping. "
        "This indicates index metadata corpus_id is not aligned with qrels corpus_id."
    )

In [ ]:
# ==============================================================================
# Shared utilities: doc matrix builder, MaxSim, metrics, latency tracker
# ==============================================================================

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")


def build_doc_matrix(embeddings, device):
    """
    Convert list of np.ndarray embeddings to a padded (n_docs, max_len, D) tensor.
    Returns (doc_matrix, doc_mask).
    """
    arrays   = [torch.from_numpy(e).float() for e in embeddings]
    max_len  = max(a.shape[0] for a in arrays)
    D        = arrays[0].shape[1]
    n        = len(arrays)
    mat      = torch.zeros(n, max_len, D, dtype=torch.float32)
    mask     = torch.zeros(n, max_len, dtype=torch.bool)
    for i, a in enumerate(arrays):
        L = a.shape[0]
        mat[i, :L] = F.normalize(a, dim=-1)
        mask[i, :L] = True
    return mat.to(device), mask.to(device)


@torch.no_grad()
def fast_maxsim(q_norm, doc_matrix, doc_mask):
    """
    q_norm   : (N_q, D)   — query tokens, L2-normalized
    doc_matrix: (n_docs, max_len, D)
    doc_mask : (n_docs, max_len)  bool
    Returns  : (N_q, n_docs)  per-token MaxSim scores
    """
    sim = torch.einsum('qd,nld->qnl', q_norm, doc_matrix)   # (N_q, n_docs, max_len)
    sim.masked_fill_(~doc_mask.unsqueeze(0), float('-inf'))
    return sim.max(dim=-1).values                             # (N_q, n_docs)


# ---------- Metrics ----------

def compute_ndcg(ranked, gt, k):
    dcg  = sum(1.0 / np.log2(r + 2) for r, i in enumerate(ranked[:k]) if i in gt)
    idcg = sum(1.0 / np.log2(r + 2) for r in range(min(len(gt), k)))
    return dcg / idcg if idcg > 0 else 0.0

def first_hit(top_k, gt):
    for r, i in enumerate(top_k):
        if i in gt: return r + 1
    return -1

def hit_metrics(top10, gt):
    h = first_hit(top10, gt)
    return {
        'r1':  int(h != -1 and h <= 1),
        'r5':  int(h != -1 and h <= 5),
        'r10': int(h != -1 and h <= 10),
        'n1':  float(compute_ndcg(top10, gt, 1)),
        'n5':  float(compute_ndcg(top10, gt, 5)),
        'n10': float(compute_ndcg(top10, gt, 10)),
    }

def _init_metric():
    return {'r1': 0, 'r5': 0, 'r10': 0, 'n1': 0.0, 'n5': 0.0, 'n10': 0.0, 'count': 0}

def _add_metric(dst, src):
    dst['r1']    += int(src['r1'])
    dst['r5']    += int(src['r5'])
    dst['r10']   += int(src['r10'])
    dst['n1']    += float(src['n1'])
    dst['n5']    += float(src['n5'])
    dst['n10']   += float(src['n10'])
    dst['count'] += 1

def _ensure(store, key):
    if key not in store: store[key] = _init_metric()
    return store[key]

def record(all_metrics, all_domain_metrics, key, m, domain):
    _add_metric(_ensure(all_metrics, key), m)
    if domain not in all_domain_metrics:
        all_domain_metrics[domain] = {}
    _add_metric(_ensure(all_domain_metrics[domain], key), m)


def print_summary(all_metrics, all_domain_metrics, method_keys, title=""):
    if title:
        print(f"\n{'='*60}\n{title}\n{'='*60}")
    print(f"{'Method':<35} {'R@1':>7} {'R@5':>7} {'R@10':>7} {'nDCG@10':>9}")
    print("-" * 65)
    for key in method_keys:
        if key not in all_metrics: continue
        m = all_metrics[key]; cnt = m['count'] or 1
        print(f"{key:<35} {m['r1']/cnt*100:6.2f}%  {m['r5']/cnt*100:6.2f}%  "
              f"{m['r10']/cnt*100:6.2f}%  {m['n10']/cnt:8.4f}")


# ---------- Latency Tracker ----------

class LatencyTracker:
    """
    Tracks per-ratio latency of MaxSim scoring AFTER pooling/pruning.

    Only measures the time for: MaxSim scoring + aggregation + top-k
    on the already-reduced multi-vector representation.
    Excludes model forward pass, pooling, and pruning computation.

    Usage:
        tracker = LatencyTracker("Hierarchical Ward Pool")
        tracker.add_ratio(ratio, score_ms)   # inside loop, per ratio
        tracker.report()                     # after loop
    """
    def __init__(self, method_name: str):
        self.name = method_name
        self.ratio_ms = {}   # ratio -> list[float]  pool+retrieve ms per query

    def add_ratio(self, ratio: float, score_ms: float):
        """Record scoring latency for a single (ratio, query) observation."""
        if ratio not in self.ratio_ms:
            self.ratio_ms[ratio] = []
        self.ratio_ms[ratio].append(score_ms)

    def report(self):
        if not self.ratio_ms:
            print(f"[{self.name}] No latency data collected.")
            return
        print(f"\n{'='*70}")
        print(f"Latency Report — {self.name}  (scoring only, post-pooling/pruning)")
        print(f"{'='*70}")
        print(f"  {'Ratio':<10} {'n':>6} {'avg ms':>10} {'p50 ms':>10} {'p95 ms':>10}")
        print(f"  {'-'*50}")
        for ratio in sorted(self.ratio_ms.keys()):
            vals = self.ratio_ms[ratio]
            n    = len(vals)
            avg  = np.mean(vals)
            p50  = np.percentile(vals, 50)
            p95  = np.percentile(vals, 95)
            print(f"  {ratio:<10.0%} {n:>6} {avg:>10.2f} {p50:>10.2f} {p95:>10.2f}")

    def to_dict(self):
        rows = []
        for ratio in sorted(self.ratio_ms.keys()):
            vals = self.ratio_ms[ratio]
            n    = len(vals)
            rows.append({
                'method':           self.name,
                'ratio':            ratio,
                'n_queries':        n,
                'avg_score_ms':     round(np.mean(vals), 3) if n else 0,
                'p50_score_ms':     round(np.percentile(vals, 50), 3) if n else 0,
                'p95_score_ms':     round(np.percentile(vals, 95), 3) if n else 0,
            })
        return rows


# ==============================================================================
# Spherical KMeans (cosine similarity) — Method 5
# Input : X (N, D) L2-normalized query token vectors
# Output: centroids (K, D) — mean-pool each cluster then re-normalize
# ==============================================================================

def spherical_kmeans(X, K, n_iters=KMEANS_ITERS):
    """
    Cluster N query token vectors into K representative centroids
    using cosine distance (spherical KMeans).

    Args:
        X      : (N, D)  L2-normalized query token vectors
        K      : int     number of clusters (representative tokens to keep)
        n_iters: int     number of Lloyd-style iterations

    Returns:
        centroids: (K, D)  L2-normalized cluster centroids
    """
    N, D = X.shape
    K = min(K, N)

    if K >= N:
        return X.clone()   # every token is its own representative

    # Initialise: pick K distinct tokens at random
    perm      = torch.randperm(N, device=X.device)
    centroids = X[perm[:K]].clone()   # (K, D)

    for _ in range(n_iters):
        # Assignment: cosine sim = dot product when X is normalised
        sim         = torch.mm(X, centroids.t())   # (N, K)
        cluster_ids = sim.argmax(dim=1)             # (N,)

        # Update: mean-pool members → re-normalize
        new_centroids = torch.zeros_like(centroids)
        for c in range(K):
            members = (cluster_ids == c).nonzero(as_tuple=True)[0]
            new_centroids[c] = (X[members].mean(dim=0)
                                if members.numel() > 0 else centroids[c])
        centroids = F.normalize(new_centroids, dim=-1)

    return centroids   # (K, D)


# ==============================================================================
# Random Token Pruning — Method 6
# Randomly sample round(M * ratio) content tokens; average over N_RANDOM_SEEDS.
# ==============================================================================

def random_prune_topk(q_norm, content_idx, doc_matrix, doc_mask,
                      topk_ratios, n_seeds=N_RANDOM_SEEDS):
    """
    For each ratio r keep k = round(M * r) randomly selected tokens.
    Scores are averaged over `n_seeds` independent random draws to reduce
    variance.

    Timing covers the full scoring work per ratio:
      random sampling + MaxSim + sum  (averaged over n_seeds).
    This matches how all other methods measure latency.

    Args:
        q_norm      : (M, D)   content token vectors, L2-normalized
        content_idx : (M,)     positions of content tokens in the full sequence
                               (unused here; kept for API symmetry)
        doc_matrix  : (n_docs, max_len, D)
        doc_mask    : (n_docs, max_len)  bool
        topk_ratios : list[float]  fractions of tokens to KEEP
        n_seeds     : int  number of random samples to average

    Returns:
        results : dict {ratio: scores (n_docs,)}
        timing  : dict {ratio: score_ms}   ms for sampling + MaxSim per ratio
    """
    M      = q_norm.shape[0]
    n_docs = doc_matrix.shape[0]
    device = q_norm.device

    if M == 0:
        zero = torch.zeros(n_docs, device=device)
        return {r: zero for r in topk_ratios}, {r: 0.0 for r in topk_ratios}

    results = {}
    timing  = {}

    for ratio in topk_ratios:
        k = max(1, round(M * ratio))
        k = min(k, M)

        # ── Time: random sample + MaxSim + sum (all scoring work) ────────
        torch.cuda.synchronize()
        t0 = time.perf_counter()

        if k == M:
            # Keep all tokens — no pruning needed; score as-is
            scores = fast_maxsim(q_norm, doc_matrix, doc_mask).sum(dim=0)
        else:
            accumulated = torch.zeros(n_docs, device=device, dtype=torch.float32)
            for _ in range(n_seeds):
                perm        = torch.randperm(M, device=device)[:k]
                pruned      = q_norm[perm]          # (k, D)
                accumulated += fast_maxsim(pruned, doc_matrix, doc_mask).sum(dim=0)
            scores = accumulated / n_seeds

        torch.cuda.synchronize()
        timing[ratio] = (time.perf_counter() - t0) * 1000.0
        # ─────────────────────────────────────────────────────────────────

        results[ratio] = scores

    return results, timing


print("Utility functions ready.")

In [ ]:
# ==============================================================================
# ALIGNMENT DEBUG — diagnose GT/index mismatch on a small sample
# ==============================================================================

print(">>> ALIGNMENT DEBUG: GT rank audit")

DEBUG_SAMPLE_SIZE = 100
DEBUG_SAMPLE_SEED = 123

if len(qa_pairs) == 0:
    raise ValueError("qa_pairs is empty")

# Reuse or build doc matrix
if 'doc_matrix' not in globals() or 'doc_mask' not in globals():
    print("Building doc matrix for debug...")
    doc_matrix, doc_mask = build_doc_matrix(all_page_embeddings, device)

n_docs = doc_matrix.shape[0]
rng = np.random.default_rng(DEBUG_SAMPLE_SEED)
sel = rng.choice(len(qa_pairs), size=min(DEBUG_SAMPLE_SIZE, len(qa_pairs)), replace=False)
debug_pairs = [qa_pairs[i] for i in sel]

invalid_gt = 0
empty_gt = 0
hit10 = 0
best_ranks = []
gt_sizes = []
docname_overlap = 0

# Normalized embedded doc names for overlap checks
if 'join_doc_name' in embedded_rows.columns:
    emb_doc_norm = embedded_rows['join_doc_name'].astype(str).str.replace('\\\\', '/', regex=False).str.replace('.pdf', '', regex=False).str.lower()
else:
    emb_doc_norm = pd.Series([''] * len(embedded_rows))

for item in tqdm(debug_pairs, desc="Alignment debug"):
    gt_raw = item.get('gt_embed_indices', [])
    gt = sorted(set(int(i) for i in gt_raw if 0 <= int(i) < n_docs))

    if len(gt_raw) == 0:
        empty_gt += 1
        continue
    if len(gt) != len(set(gt_raw)):
        invalid_gt += 1
    if len(gt) == 0:
        continue

    gt_sizes.append(len(gt))

    # doc_name overlap sanity
    q_doc = str(item.get('doc_name', '')).replace('\\', '/').replace('.pdf', '').lower().strip()
    if q_doc:
        gt_docs = emb_doc_norm.iloc[gt].tolist()
        if any((q_doc == d) or (q_doc in d) or (d in q_doc) for d in gt_docs if d):
            docname_overlap += 1

    q_inputs = query_processor.process_queries([item['question']]).to(device)
    if 'token_type_ids' not in q_inputs and 'input_ids' in q_inputs:
        q_inputs['token_type_ids'] = torch.zeros_like(q_inputs['input_ids'])

    with torch.no_grad():
        q_proj = query_model(**q_inputs)

    if hasattr(q_proj, 'last_hidden_state') and q_proj.last_hidden_state is not None:
        q_proj = q_proj.last_hidden_state
    elif isinstance(q_proj, (tuple, list)) and len(q_proj) > 0:
        q_proj = q_proj[0]

    attn_mask = q_inputs['attention_mask'][0]
    q_idx = torch.where(attn_mask > 0)[0]
    q_norm = F.normalize(q_proj[0][q_idx].float(), dim=-1)

    scores = fast_maxsim(q_norm, doc_matrix, doc_mask).sum(dim=0).detach().float().cpu()
    top10 = torch.topk(scores, min(10, n_docs)).indices.tolist()

    if any(i in gt for i in top10):
        hit10 += 1

    gt_scores = scores[gt]
    best_gt = float(torch.max(gt_scores).item())
    rank = int((scores > best_gt).sum().item()) + 1
    best_ranks.append(rank)

print("\n--- Alignment debug summary ---")
print(f"Queries checked              : {len(debug_pairs)}")
print(f"Empty GT rows               : {empty_gt}")
print(f"Rows with invalid GT idx    : {invalid_gt}")
print(f"Hit@10 (debug sample)       : {hit10 / max(1, len(debug_pairs)) * 100:.2f}%")

if gt_sizes:
    print(f"GT size stats               : min={min(gt_sizes)}, p50={int(np.median(gt_sizes))}, p95={int(np.percentile(gt_sizes, 95))}, max={max(gt_sizes)}")
if best_ranks:
    print(f"Best GT rank stats          : p50={int(np.median(best_ranks))}, p90={int(np.percentile(best_ranks, 90))}, p95={int(np.percentile(best_ranks, 95))}")
print(f"Doc-name overlap in GT      : {docname_overlap}/{max(1, len(debug_pairs))}")

print("\nNếu Best GT rank vẫn rất cao (p50/p90 lớn), khả năng cao query encoding hoặc mapping GT vẫn lệch.")

In [ ]:
# ==============================================================================
# METHOD 1 — Traditional MaxSim
# Queries encoded live with ColPali (plain forward, no attention capture).
# ==============================================================================

print(">>> METHOD 1: Traditional MaxSim")

trad_metrics        = {}
trad_domain_metrics = {}
trad_query_rows     = []
trad_latency        = LatencyTracker("Traditional MaxSim")

METHOD_KEYS_TRAD = ['traditional']

# Build full doc matrix from all page embeddings
print(f"Building doc matrix from {len(all_page_embeddings)} page embeddings...")
doc_matrix, doc_mask = build_doc_matrix(all_page_embeddings, device)
n_docs = doc_matrix.shape[0]
print(f"Doc matrix shape: {doc_matrix.shape}")

pbar = tqdm(enumerate(qa_pairs), total=len(qa_pairs), desc="Traditional MaxSim")

for q_idx, item in pbar:
    question = item['question']
    gt_set   = set(item['gt_embed_indices'])
    domain   = item['domain']

    # Encode query live
    q_inputs = query_processor.process_queries([question]).to(device)
    if 'token_type_ids' not in q_inputs and 'input_ids' in q_inputs:
        q_inputs['token_type_ids'] = torch.zeros_like(q_inputs['input_ids'])

    with torch.no_grad():
        q_proj = query_model(**q_inputs)

    if hasattr(q_proj, "last_hidden_state") and q_proj.last_hidden_state is not None:
        q_proj = q_proj.last_hidden_state
    elif isinstance(q_proj, (tuple, list)) and len(q_proj) > 0:
        q_proj = q_proj[0]

    attn_mask = q_inputs['attention_mask'][0]
    trad_idx  = torch.where(attn_mask > 0)[0]
    q_emb     = q_proj[0][trad_idx].float()
    q_norm    = F.normalize(q_emb, dim=-1)

    # Retrieval (timed — 100% baseline)
    torch.cuda.synchronize()
    t_score_start = time.perf_counter()

    M      = fast_maxsim(q_norm, doc_matrix, doc_mask)
    scores = M.sum(dim=0)
    top10  = torch.topk(scores, min(10, n_docs)).indices.cpu().tolist()

    torch.cuda.synchronize()
    score_ms = (time.perf_counter() - t_score_start) * 1000.0
    trad_latency.add_ratio(1.0, score_ms)

    m = hit_metrics(top10, gt_set)
    record(trad_metrics, trad_domain_metrics, 'traditional', m, domain)

    trad_query_rows.append({
        'query_id':       q_idx,
        'doc_name':       item['doc_name'],
        'domain':         domain,
        'question':       question,
        'trad_r@1':       m['r1'],
        'trad_r@5':       m['r5'],
        'trad_r@10':      m['r10'],
        'trad_ndcg@1':    round(m['n1'],  4),
        'trad_ndcg@5':    round(m['n5'],  4),
        'trad_ndcg@10':   round(m['n10'], 4),
    })

print_summary(trad_metrics, trad_domain_metrics, METHOD_KEYS_TRAD,
              title="Traditional MaxSim Results")
trad_latency.report()

pd.DataFrame(trad_query_rows).to_csv(
    os.path.join(WORKING_DIR, "traditional_queries.csv"), index=False)
print("\n✅ Saved: traditional_queries.csv")

In [ ]:
# ==============================================================================
# METHOD 7: RVQ Retrieval — Pre-trained Codebooks (NO re-training)
#
# Pre-trained codebooks (upload lên kaggle dataset):
#   /kaggle/input/datasets/namthi/weight-train-rvq-vidore/c8_f32_codebook.npy
#   /kaggle/input/datasets/namthi/weight-train-rvq-vidore/c16_f32_codebook.npy
#
# Codebook format: np.ndarray (NQ, CB_SIZE, EMB_DIM) float32, L2-normalized
#   c8_f32  -> (8,  32, 128)   8  bytes/patch  (64× vs float32)
#   c16_f32 -> (16, 32, 128)   16 bytes/patch  (32× vs float32)
#
# ── PART A: Pure RVQ — 2-stage greedy ADC ──────────────────────────────────
#   Stage 1 (c8):  full-scan ADC → top-K candidates
#   Stage 2 (c16): re-rank candidates → top-10
#
# ── PART B: RVQ + Beam Search (b=5) ────────────────────────────────────────
#   Stage 1 (c8):  same greedy ADC full-scan → top-K candidates  (fast)
#   Stage 2 (c16): re-rank with BEAM SEARCH b=5 on query tokens
#
#   Beam search explanation:
#     Standard ADC uses the RAW query token vectors to compute similarity with
#     codebook-indexed document patches.  Beam Search generates b=5 candidate
#     RECONSTRUCTED query vectors per token (by exploring the top-b paths in
#     the RVQ codebook tree), then takes the MAX similarity across all b probes.
#     This improves recall because quantisation errors in the document index
#     are partially compensated by checking multiple "nearby" query representations.
#
# Requires (from earlier cells):
#   all_page_embeddings  : list of np.ndarray (L, D)
#   qa_pairs, query_model, query_processor, device
#   hit_metrics, record, print_summary, _init_metric, WORKING_DIR
# ==============================================================================

import os, gc, time
import numpy as np
import torch
import torch.nn.functional as F
from tqdm.notebook import tqdm
import pandas as pd

# ── GPU settings ──────────────────────────────────────────────────────────────
device = "cuda" if torch.cuda.is_available() else "cpu"
if torch.cuda.is_available():
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32       = True
    torch.backends.cudnn.benchmark        = True
    print(f"  GPU : {torch.cuda.get_device_name(0)}")
    print(f"  VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

# ══════════════════════════════════════════════════════════════════════════════
# CONFIG
# ══════════════════════════════════════════════════════════════════════════════
CODEBOOK_ROOT    = "/kaggle/input/datasets/namthi/weight-train-rvq-vidore"
QUANT_PAGE_BATCH = 128     # pages per GPU batch during corpus quantisation
ADC_CHUNK_SIZE   = 4096    # documents per chunk during full-scan ADC
BEAM_WIDTH       = 5       # beam search width (Part B)

# Pre-trained codebook metadata
CB_META = {
    "c8_f32":  {"NQ":  8, "CB_SIZE": 32, "file": "c8_f32_codebook.npy"},
    "c16_f32": {"NQ": 16, "CB_SIZE": 32, "file": "c16_f32_codebook.npy"},
}

# Sweep: (coarse_label, fine_label, TOP_K_stage1)
RVQ_SWEEP = [
    ("c8_f32", "c16_f32", 100),
    ("c8_f32", "c16_f32", 500),
    ("c8_f32", "c16_f32", 1000),
]

print(">>>> METHOD 7: RVQ Retrieval (Pre-trained Codebooks)")
print(f"  Codebook root : {CODEBOOK_ROOT}")
print(f"  Configs       : {list(CB_META.keys())}")
print(f"  Sweep TOP-K   : {[k for _,_,k in RVQ_SWEEP]}")
print(f"  Beam width    : {BEAM_WIDTH}  (Part B only)")
print()

# ══════════════════════════════════════════════════════════════════════════════
# STEP 1: Load pre-trained codebooks from .npy files
# ══════════════════════════════════════════════════════════════════════════════
def load_codebook(label):
    """
    Load .npy → list of NQ GPU tensors, each shape (CB_SIZE, EMB_DIM).
    File on disk: (NQ, CB_SIZE, EMB_DIM) float32.
    """
    cfg     = CB_META[label]
    path    = os.path.join(CODEBOOK_ROOT, cfg["file"])
    cbs_np  = np.load(path)                              # (NQ, CB_SIZE, D)
    assert cbs_np.ndim == 3, f"Expected 3D array, got {cbs_np.shape}"
    assert cbs_np.shape[0] == cfg["NQ"],     f"NQ mismatch: {cbs_np.shape[0]} vs {cfg['NQ']}"
    assert cbs_np.shape[1] == cfg["CB_SIZE"],f"CB_SIZE mismatch"
    # Ensure L2-normalised (should already be, but safety check)
    norms   = np.linalg.norm(cbs_np, axis=-1, keepdims=True).clip(min=1e-8)
    cbs_np  = cbs_np / norms
    cbs     = [torch.from_numpy(cbs_np[qi].copy()).float().to(device)
               for qi in range(cbs_np.shape[0])]
    print(f"  {label}: {path}")
    print(f"    shape={cbs_np.shape}  |  {cbs_np.nbytes/1e6:.2f} MB")
    return cbs

print("Loading codebooks...")
cb_cache = {}
for lbl in CB_META:
    cb_cache[lbl] = load_codebook(lbl)
print("✅ Codebooks loaded.\n")

# ══════════════════════════════════════════════════════════════════════════════
# STEP 2: Sanity-check all_page_embeddings + compute index stats
# ══════════════════════════════════════════════════════════════════════════════
if 'all_page_embeddings' not in globals() or all_page_embeddings is None:
    raise RuntimeError(
        "all_page_embeddings not found — run the index-load cell first."
    )

n_pages_total       = len(all_page_embeddings)
EMB_DIM             = int(all_page_embeddings[0].shape[-1])
doc_lengths         = [e.shape[0] for e in all_page_embeddings]
max_doc_len         = int(max(doc_lengths))
total_patches       = int(sum(doc_lengths))
bytes_per_patch_f32 = EMB_DIM * 4

print(f"Document index:")
print(f"  Pages        : {n_pages_total:,}")
print(f"  Total patches: {total_patches:,}")
print(f"  EMB_DIM      : {EMB_DIM}   max_doc_len: {max_doc_len}")
print(f"  float32 size : {total_patches * bytes_per_patch_f32 / 1e6:.1f} MB\n")

# ══════════════════════════════════════════════════════════════════════════════
# STEP 3: Quantise corpus with each codebook (greedy nearest-neighbour RVQ)
# ══════════════════════════════════════════════════════════════════════════════
@torch.no_grad()
def quantize_corpus(codebooks, label):
    """
    Encode every document patch using greedy RVQ:
      residual[0] = patch_vector
      for each level qi:
          best_idx   = argmax cosine_sim(residual, codebook[qi])
          residual  -= codebook[qi][best_idx]
    Returns:
      idx_arr  : (n_docs, max_doc_len, NQ)  uint8  — codebook indices
      mask_arr : (n_docs, max_doc_len)       bool   — valid-patch mask
    """
    NQ       = len(codebooks)
    n        = n_pages_total
    idx_arr  = np.zeros((n, max_doc_len, NQ), dtype=np.uint8)
    mask_arr = np.zeros((n, max_doc_len),      dtype=bool)

    for bs in tqdm(range(0, n, QUANT_PAGE_BATCH),
                   desc=f"  Quantise {label}", leave=False):
        be    = min(bs + QUANT_PAGE_BATCH, n)
        pages = all_page_embeddings[bs:be]
        B     = len(pages)

        # Build padded batch (B, max_doc_len, D)
        bt = torch.zeros(B, max_doc_len, EMB_DIM, device=device)
        bm = torch.zeros(B, max_doc_len, dtype=torch.bool)
        for i, emb in enumerate(pages):
            L         = emb.shape[0]
            emb_t     = torch.from_numpy(emb.astype(np.float32)).to(device)
            bt[i, :L] = F.normalize(emb_t, dim=-1)
            bm[i, :L] = True

        flat     = bt.reshape(B * max_doc_len, EMB_DIM)   # (B*L, D)
        residual = flat.clone()
        idxs     = torch.zeros(B * max_doc_len, NQ, dtype=torch.long, device=device)

        for qi, cb in enumerate(codebooks):                # cb: (CB, D)
            sims       = torch.mm(residual, cb.t())        # (B*L, CB)
            best_idx   = sims.argmax(dim=-1)               # (B*L,)
            residual  -= cb[best_idx]                      # subtract nearest entry
            idxs[:, qi] = best_idx

        idxs = idxs.reshape(B, max_doc_len, NQ)
        idx_arr[bs:be]  = idxs.cpu().numpy().astype(np.uint8)
        mask_arr[bs:be] = bm.numpy()

    return (torch.from_numpy(idx_arr).pin_memory(),
            torch.from_numpy(mask_arr).pin_memory())


print("Quantising corpus for each codebook config...")
rvq_idx = {}
for lbl in CB_META:
    t0 = time.time()
    rvq_idx[lbl] = quantize_corpus(cb_cache[lbl], lbl)
    mb = rvq_idx[lbl][0].element_size() * rvq_idx[lbl][0].nelement() / 1e6
    print(f"  {lbl}: {time.time()-t0:.1f}s | {mb:.1f} MB uint8  "
          f"(vs {total_patches * bytes_per_patch_f32 / 1e6:.0f} MB float32)")

# Free memory since we only need the quantised indices (rvq_idx) from now on
if 'all_page_embeddings' in globals():
    del all_page_embeddings

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print("✅ Corpus quantisation done (deleted all_page_embeddings to free RAM).\n")


# ══════════════════════════════════════════════════════════════════════════════
# ADC (Asymmetric Distance Computation) HELPERS
# ══════════════════════════════════════════════════════════════════════════════

@torch.no_grad()
def _build_lut(q_norm, codebooks):
    """LUT[qi, tok_i, cb_entry] = dot(query_token_i, codebook[qi][cb_entry])
    Shape: (NQ, Nq, CB_SIZE)"""
    return torch.stack([torch.mm(q_norm, cb.t()) for cb in codebooks])


@torch.no_grad()
def adc_full_scan(q_norm, lut, idx_cpu, mask_cpu):
    """
    Full-corpus ADC scan.
    Returns: (n_docs,) scores tensor on GPU.
    """
    n_docs = idx_cpu.shape[0]
    NQ_    = idx_cpu.shape[2]
    Nq     = q_norm.shape[0]
    scores = torch.zeros(n_docs, device=device)

    for s in range(0, n_docs, ADC_CHUNK_SIZE):
        e   = min(s + ADC_CHUNK_SIZE, n_docs)
        ci  = idx_cpu[s:e].to(device, non_blocking=True)   # (B, L, NQ)
        cm  = mask_cpu[s:e].to(device, non_blocking=True)  # (B, L)
        cs, ml = ci.shape[0], ci.shape[1]

        # sim[tok_i, doc_j, patch_k] = sum_qi lut[qi, tok_i, idx(doc_j, patch_k, qi)]
        sim = torch.zeros(Nq, cs, ml, device=device)
        for q in range(NQ_):
            idx_e = ci[:, :, q].long().unsqueeze(0).expand(Nq, -1, -1)
            sim  += torch.gather(lut[q].unsqueeze(1).expand(-1, cs, -1), 2, idx_e)

        sim.masked_fill_(~cm.unsqueeze(0), float('-inf'))
        # MaxSim: max over patches, sum over query tokens
        scores[s:e] = sim.max(dim=-1).values.sum(dim=0)

    return scores


@torch.no_grad()
def adc_rerank(q_norm, lut, candidates, idx_cpu, mask_cpu):
    """
    ADC re-ranking on a candidate subset.
    Returns: top-10 global doc indices.
    """
    NQ_  = idx_cpu.shape[2]
    Nq   = q_norm.shape[0]
    K    = len(candidates)
    ct   = torch.tensor(candidates, dtype=torch.long)
    ci   = idx_cpu[ct].to(device)    # (K, L, NQ)
    cm   = mask_cpu[ct].to(device)   # (K, L)
    ml   = ci.shape[1]
    sim  = torch.zeros(Nq, K, ml, device=device)
    for q in range(NQ_):
        idx_e = ci[:, :, q].long().unsqueeze(0).expand(Nq, -1, -1)
        sim  += torch.gather(lut[q].unsqueeze(1).expand(-1, K, -1), 2, idx_e)
    sim.masked_fill_(~cm.unsqueeze(0), float('-inf'))
    fine = sim.max(dim=-1).values.sum(dim=0)   # (K,)
    top  = torch.topk(fine, min(10, K)).indices.cpu().tolist()
    return [candidates[i] for i in top]


# ══════════════════════════════════════════════════════════════════════════════
# BEAM SEARCH HELPERS  (Part B)
# ══════════════════════════════════════════════════════════════════════════════

@torch.no_grad()
def _beam_reconstruct(q_norm, codebooks, beam_width):
    """
    True beam search through RVQ codebook tree for all query tokens.

    Algorithm (batched over all Nq tokens simultaneously):
      Level 0 : for each token, pick top-B codebook entries → B candidate paths
      Level 1+: for each of B paths, pick the BEST remaining entry among all CB
                entries → keep best B paths (by cumulative inner-product score)

    Returns: (Nq, B, D) float32 tensor — B L2-normalised reconstructed vectors
             per query token (on GPU).
    """
    Nq, D  = q_norm.shape
    B      = beam_width
    NQ     = len(codebooks)

    # ── Level 0: expand to B beams ──────────────────────────────────────────
    cb0     = codebooks[0]                                            # (CB, D)
    sims0   = torch.mm(q_norm, cb0.t())                               # (Nq, CB)
    tvals0, tidxs0 = torch.topk(sims0, min(B, cb0.shape[0]), dim=-1) # (Nq, B)

    # residuals[nq, b] = q_norm[nq] - cb0[tidxs0[nq,b]]
    residuals = q_norm.unsqueeze(1) - cb0[tidxs0]                    # (Nq, B, D)
    recons    = cb0[tidxs0].clone()                                   # (Nq, B, D)
    scores    = tvals0                                                # (Nq, B)

    arange_nq = torch.arange(Nq, device=device)

    # ── Levels 1 .. NQ-1: extend & prune ────────────────────────────────────
    for qi_level in range(1, NQ):
        cb  = codebooks[qi_level]                                     # (CB, D)
        CB  = cb.shape[0]

        # Compute sims for all (Nq, B) residuals against CB entries
        res_flat = residuals.reshape(Nq * B, D)                       # (Nq*B, D)
        sims     = torch.mm(res_flat, cb.t()).reshape(Nq, B, CB)      # (Nq, B, CB)

        # Candidate scores: current_path_score + sim_for_new_entry
        cand_scores = scores.unsqueeze(-1) + sims                     # (Nq, B, CB)
        cand_flat   = cand_scores.reshape(Nq, B * CB)                 # (Nq, B*CB)

        # Prune: keep top-B candidates
        top_vals, top_flat_idxs = torch.topk(cand_flat, B, dim=-1)   # (Nq, B)
        beam_from = top_flat_idxs // CB                               # parent beam
        cb_chosen = top_flat_idxs %  CB                               # CB entry

        # Gather parent states and extend
        # par_res/par_recon: (Nq, B, D)
        par_res   = residuals[arange_nq.unsqueeze(1), beam_from]      # (Nq, B, D)
        par_recon = recons[arange_nq.unsqueeze(1), beam_from]         # (Nq, B, D)
        cb_vecs   = cb[cb_chosen]                                     # (Nq, B, D)

        residuals = par_res - cb_vecs
        recons    = par_recon + cb_vecs
        scores    = top_vals

    return F.normalize(recons, dim=-1)   # (Nq, B, D)


@torch.no_grad()
def beam_adc_rerank(q_norm, candidates, idx_cpu, mask_cpu, codebooks,
                    beam_width=BEAM_WIDTH):
    """
    Beam-search ADC re-ranking on candidate subset.

    For each query token, we have B reconstructed vectors (beam_width paths).
    Similarity between query token i and doc patch is:
        MAX over b=0..B-1 of [ sum_qi lut[qi, i*B+b, patch_idx_qi] ]
    where the MAX captures the best-matching beam probe.

    Returns: top-10 global doc indices.
    """
    Nq  = q_norm.shape[0]
    B   = beam_width
    NQ_ = idx_cpu.shape[2]
    K   = len(candidates)

    # Build B reconstructed query vectors per token: (Nq, B, D)
    beam_q = _beam_reconstruct(q_norm, codebooks, B)         # (Nq, B, D)
    beam_q_flat = beam_q.reshape(Nq * B, -1)                 # (Nq*B, D)

    # LUT: (NQ, Nq*B, CB)
    lut = torch.stack([torch.mm(beam_q_flat, cb.t()) for cb in codebooks])

    ct  = torch.tensor(candidates, dtype=torch.long)
    ci  = idx_cpu[ct].to(device)    # (K, L, NQ)
    cm  = mask_cpu[ct].to(device)   # (K, L)
    ml  = ci.shape[1]

    # sim: (Nq*B, K, L)
    # For large K/ml, chunk over candidates to avoid OOM
    CAND_CHUNK = 256
    fine = torch.zeros(K, device=device)
    for cs in range(0, K, CAND_CHUNK):
        ce   = min(cs + CAND_CHUNK, K)
        ci_c = ci[cs:ce]       # (Cc, L, NQ)
        cm_c = cm[cs:ce]       # (Cc, L)
        Cc   = ci_c.shape[0]

        sim_c = torch.zeros(Nq * B, Cc, ml, device=device)
        for q in range(NQ_):
            idx_e = ci_c[:, :, q].long().unsqueeze(0).expand(Nq * B, -1, -1)
            sim_c += torch.gather(
                lut[q].unsqueeze(1).expand(-1, Cc, -1), 2, idx_e
            )
        sim_c.masked_fill_(~cm_c.unsqueeze(0), float('-inf'))   # (Nq*B, Cc, ml)

        # Reshape → (Nq, B, Cc, ml), max over B → (Nq, Cc, ml)
        sim_c = sim_c.view(Nq, B, Cc, ml).max(dim=1).values

        # MaxSim: max over patches → (Nq, Cc), sum over tokens → (Cc,)
        fine[cs:ce] = sim_c.max(dim=-1).values.sum(dim=0)

    top  = torch.topk(fine, min(10, K)).indices.cpu().tolist()
    return [candidates[i] for i in top]


# ══════════════════════════════════════════════════════════════════════════════
# PART A: PURE RVQ — 2-Stage Greedy ADC
# ══════════════════════════════════════════════════════════════════════════════
print("=" * 70)
print("PART A: RVQ — 2-Stage Greedy ADC")
print("=" * 70)

rvq_met_a   = {}
rvq_dom_a   = {}
rvq_rows_a  = []
_lat_a      = {}

for lbl_c, lbl_f, TOP_K in RVQ_SWEEP:
    cfg_lbl = f"rvq_{lbl_c}_{lbl_f}_k{TOP_K}"
    cb_c    = cb_cache[lbl_c]
    cb_f    = cb_cache[lbl_f]
    idx_c, mask_c = rvq_idx[lbl_c]
    idx_f, mask_f = rvq_idx[lbl_f]
    mem_c   = total_patches * CB_META[lbl_c]["NQ"] / 1e6
    mem_f   = total_patches * CB_META[lbl_f]["NQ"] / 1e6

    print(f"\n  {cfg_lbl}  (Stage1 {mem_c:.0f}MB → top-{TOP_K} → Stage2 {mem_f:.0f}MB)")
    t_eval = time.time()

    for qi, item in tqdm(enumerate(qa_pairs), total=len(qa_pairs),
                         desc=f"  {cfg_lbl}", leave=False):
        q_in = query_processor.process_queries([item['question']]).to(device)
        with torch.no_grad():
            q_proj = query_model(**q_in)
        tidx   = torch.where(q_in['attention_mask'][0] > 0)[0]
        q_norm = F.normalize(q_proj[0][tidx].float(), dim=-1)
        gt_set = item.get('gt_relevance', item['gt_embed_indices'])
        domain = item['domain']

        if torch.cuda.is_available(): torch.cuda.synchronize()
        t0 = time.perf_counter()

        # Stage 1: greedy coarse ADC → top-K
        lut_c    = _build_lut(q_norm, cb_c)
        scores_c = adc_full_scan(q_norm, lut_c, idx_c, mask_c)
        cands    = torch.topk(scores_c, min(TOP_K, n_pages_total)).indices.cpu().tolist()

        # Stage 2: greedy fine ADC on candidates → top-10
        lut_f = _build_lut(q_norm, cb_f)
        top10 = adc_rerank(q_norm, lut_f, cands, idx_f, mask_f)

        if torch.cuda.is_available(): torch.cuda.synchronize()
        ms = (time.perf_counter() - t0) * 1000
        _lat_a.setdefault(cfg_lbl, []).append(ms)

        m = hit_metrics(top10, gt_set)
        record(rvq_met_a, rvq_dom_a, cfg_lbl, m, domain)
        rvq_rows_a.append({
            'method': 'RVQ', 'part': 'A',
            'config': cfg_lbl, 'coarse': lbl_c, 'fine': lbl_f,
            'top_K': TOP_K, 'beam_width': 1,
            'query_id': qi, 'doc_name': item['doc_name'],
            'domain': domain, 'question': item['question'],
            'r@1': m['r1'], 'r@5': m['r5'], 'r@10': m['r10'],
            'ndcg@10': round(m['n10'], 4), 'score_ms': round(ms, 3),
        })

    avg_ms = np.mean(_lat_a.get(cfg_lbl, [0.0]))
    print(f"    Done {time.time()-t_eval:.1f}s | avg {avg_ms:.1f} ms/query")

print_summary(rvq_met_a, rvq_dom_a,
              [f"rvq_{c}_{f}_k{k}" for c, f, k in RVQ_SWEEP],
              title="Part A — RVQ 2-Stage Greedy ADC")

pd.DataFrame(rvq_rows_a).to_csv(
    os.path.join(WORKING_DIR, "method7_partA_rvq.csv"), index=False)
print("✅ Saved: method7_partA_rvq.csv")


# ══════════════════════════════════════════════════════════════════════════════
# PART B: RVQ + BEAM SEARCH (b=5)
#   Stage 1: greedy coarse ADC → top-K  (same as Part A, no extra cost)
#   Stage 2: beam-search re-ranking with b=5 query probes per token
# ══════════════════════════════════════════════════════════════════════════════
print("\n" + "=" * 70)
print(f"PART B: RVQ + Beam Search  (b={BEAM_WIDTH})")
print("=" * 70)

rvq_met_b   = {}
rvq_dom_b   = {}
rvq_rows_b  = []
_lat_b      = {}

for lbl_c, lbl_f, TOP_K in RVQ_SWEEP:
    cfg_lbl = f"rvq_beam{BEAM_WIDTH}_{lbl_c}_{lbl_f}_k{TOP_K}"
    cb_c    = cb_cache[lbl_c]
    cb_f    = cb_cache[lbl_f]
    idx_c, mask_c = rvq_idx[lbl_c]
    idx_f, mask_f = rvq_idx[lbl_f]
    mem_c   = total_patches * CB_META[lbl_c]["NQ"] / 1e6
    mem_f   = total_patches * CB_META[lbl_f]["NQ"] / 1e6

    print(f"\n  {cfg_lbl}  (Stage1 greedy {mem_c:.0f}MB → top-{TOP_K}"
          f" → Stage2 beam-{BEAM_WIDTH} {mem_f:.0f}MB)")
    t_eval = time.time()

    for qi, item in tqdm(enumerate(qa_pairs), total=len(qa_pairs),
                         desc=f"  {cfg_lbl}", leave=False):
        q_in = query_processor.process_queries([item['question']]).to(device)
        with torch.no_grad():
            q_proj = query_model(**q_in)
        tidx   = torch.where(q_in['attention_mask'][0] > 0)[0]
        q_norm = F.normalize(q_proj[0][tidx].float(), dim=-1)
        gt_set = item.get('gt_relevance', item['gt_embed_indices'])
        domain = item['domain']

        if torch.cuda.is_available(): torch.cuda.synchronize()
        t0 = time.perf_counter()

        # Stage 1: greedy coarse ADC → top-K  (identical to Part A Stage 1)
        lut_c    = _build_lut(q_norm, cb_c)
        scores_c = adc_full_scan(q_norm, lut_c, idx_c, mask_c)
        cands    = torch.topk(scores_c, min(TOP_K, n_pages_total)).indices.cpu().tolist()

        # Stage 2: beam-search fine ADC re-ranking
        top10 = beam_adc_rerank(q_norm, cands, idx_f, mask_f, cb_f, BEAM_WIDTH)

        if torch.cuda.is_available(): torch.cuda.synchronize()
        ms = (time.perf_counter() - t0) * 1000
        _lat_b.setdefault(cfg_lbl, []).append(ms)

        m = hit_metrics(top10, gt_set)
        record(rvq_met_b, rvq_dom_b, cfg_lbl, m, domain)
        rvq_rows_b.append({
            'method': 'RVQ+Beam', 'part': 'B',
            'config': cfg_lbl, 'coarse': lbl_c, 'fine': lbl_f,
            'top_K': TOP_K, 'beam_width': BEAM_WIDTH,
            'query_id': qi, 'doc_name': item['doc_name'],
            'domain': domain, 'question': item['question'],
            'r@1': m['r1'], 'r@5': m['r5'], 'r@10': m['r10'],
            'ndcg@10': round(m['n10'], 4), 'score_ms': round(ms, 3),
        })

    avg_ms = np.mean(_lat_b.get(cfg_lbl, [0.0]))
    print(f"    Done {time.time()-t_eval:.1f}s | avg {avg_ms:.1f} ms/query")

print_summary(rvq_met_b, rvq_dom_b,
              [f"rvq_beam{BEAM_WIDTH}_{c}_{f}_k{k}" for c, f, k in RVQ_SWEEP],
              title=f"Part B — RVQ + Beam Search (b={BEAM_WIDTH})")

pd.DataFrame(rvq_rows_b).to_csv(
    os.path.join(WORKING_DIR, "method7_partB_rvq_beam.csv"), index=False)
print(f"✅ Saved: method7_partB_rvq_beam.csv")


# ══════════════════════════════════════════════════════════════════════════════
# COMBINED COMPARISON TABLE
# ══════════════════════════════════════════════════════════════════════════════
print("\n" + "=" * 105)
print(f"{'Config':<45} {'R@1':>7} {'R@5':>7} {'R@10':>7} {'nDCG@10':>9} {'ms/q':>8}")
print("-" * 105)

def _print_row(cfg_lbl, met_dict, lat_dict):
    m   = met_dict.get(cfg_lbl, _init_metric())
    cnt = max(m['count'], 1)
    lat = np.mean(lat_dict.get(cfg_lbl, [0.0]))
    print(f"{cfg_lbl:<45} "
          f"{m['r1']/cnt*100:6.2f}%  "
          f"{m['r5']/cnt*100:6.2f}%  "
          f"{m['r10']/cnt*100:6.2f}%  "
          f"{m['n10']/cnt:8.4f}  "
          f"{lat:7.1f}ms")

for c, f, k in RVQ_SWEEP:
    _print_row(f"rvq_{c}_{f}_k{k}",            rvq_met_a, _lat_a)
    _print_row(f"rvq_beam{BEAM_WIDTH}_{c}_{f}_k{k}", rvq_met_b, _lat_b)
    print()

# Save combined
all_rows = rvq_rows_a + rvq_rows_b
pd.DataFrame(all_rows).to_csv(
    os.path.join(WORKING_DIR, "method7_rvq_combined.csv"), index=False)
print("✅ Saved: method7_rvq_combined.csv")
print("\n>>> Method 7 complete.")
